# Automated Urdu word corpus: five-minute pilot

Continue after notebooks 01 and 02 in the same Colab runtime. Input: original source MP4 and five-minute WAV/transcript/scene candidates under `/content/urdu-av-pilot`. The accepted transcript is used unchanged. No manual annotation sheet is required.

This notebook aligns Urdu words, measures all 9,000 frames, scores audio/face correspondence with SyncNet, exports passing paired clips and an occurrence dictionary, then checks exported media. It does not produce phoneme labels, canonical speech, composed sentences, voice transfer or dubbing. Thresholds are provisional; automatic acceptance is not a measured accuracy claim.


In [ ]:
import sys, subprocess
from pathlib import Path
ROOT=Path('/content/urdu-av-pilot')
for name in ['raw/VD-RK-nY66k.mp4','pilot_0090_0390/audio_16k.wav','pilot_0090_0390/transcript_raw.json','pilot_0090_0390/scene_candidates.json']:
    assert (ROOT/name).exists(), f'Missing {name}: run acquisition/transcription first'
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','whisperx==3.8.6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','mediapipe==0.10.32','python_speech_features==0.6','nltk','pandas','imageio-ffmpeg'],check=True)


## Install project code
The following cells embed the repository scripts so private repository authentication is unnecessary. Regenerate this notebook after editing those scripts.

In [ ]:
Path('/content/pilot_pipeline.py').write_text('"""Five-minute Urdu pilot: alignment, frame checks, SyncNet, paired exports.\nAutomatic measurements use provisional thresholds; they are not human ground truth.\n"""\nfrom pathlib import Path\nimport argparse, collections, hashlib, json, math, subprocess, sys, unicodedata\n\nCONFIG = dict(min_word_seconds=0.04, max_word_seconds=2.0, min_alignment_score=0.1,\n              min_mouth_pixels=28, min_face_height=100, max_yaw_degrees=40,\n              min_mouth_laplacian=15, max_landmark_jump=0.3,\n              sync_confidence=3.0, max_sync_offset_frames=2, context_seconds=0.12, face_detection_score=0.9)\n\ndef read(p): return json.loads(Path(p).read_text())\ndef save(p, v): Path(p).write_text(json.dumps(v, ensure_ascii=False, indent=2, allow_nan=False)+\'\\n\')\ndef sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef lookup(text):\n    return \'\'.join(c for c in unicodedata.normalize(\'NFC\', text).strip() if not unicodedata.category(c).startswith(\'P\'))\ndef cut_window(start, end, duration=300, fps=30, context=.12):\n    return max(0, math.floor((start-context)*fps))/fps, min(duration, math.ceil((end+context)*fps)/fps)\ndef alignment_reasons(w):\n    a,b=w.get(\'start\'),w.get(\'end\'); reasons=[]\n    if a is None or b is None or not all(math.isfinite(x) for x in [a,b]) or not 0<=a<b<=300:\n        return [\'missing_or_invalid_alignment\']\n    if not CONFIG[\'min_word_seconds\']<=b-a<=CONFIG[\'max_word_seconds\']: reasons.append(\'word_duration\')\n    if w.get(\'score\',0)<CONFIG[\'min_alignment_score\']: reasons.append(\'alignment_score\')\n    if w.get(\'unsupported_letters\'): reasons.append(\'unsupported_letters\')\n    return reasons\n\ndef download(url, path):\n    import urllib.request\n    path=Path(path)\n    if not path.exists():\n        temp=path.with_suffix(path.suffix+\'.part\'); urllib.request.urlretrieve(url,temp);temp.replace(path)\n    return path\n\ndef align_stage(root,out):\n    import numpy as np, torch, whisperx, nltk\n    from huggingface_hub import snapshot_download\n    from scipy.io import wavfile\n    torch.set_num_threads(2)\n    nltk.download(\'punkt_tab\',quiet=True)\n    model_id=\'kingabzpro/wav2vec2-large-xls-r-300m-Urdu\'\n    model_path=snapshot_download(model_id,allow_patterns=[\'*.json\',\'*.safetensors\',\'pytorch_model.bin\'])\n    model,meta=whisperx.load_align_model(\'ur\',\'cpu\',model_name=model_path)\n    sr,wav=wavfile.read(root/\'pilot_0090_0390/audio_16k.wav\');assert sr==16000 and len(wav)==4800000\n    audio=wav.astype(np.float32)/32768\n    raw=read(root/\'pilot_0090_0390/transcript_raw.json\')\n    segments=[];words=[]; failures=[]\n    for i,s in enumerate(raw[\'segments\']):\n        # Keep accepted text unchanged; expand the acoustic search window slightly.\n        lo=max(0,s[\'start\']-.15);hi=min(300,s[\'end\']+.15)\n        result=whisperx.align([dict(start=lo,end=hi,text=s[\'text\'])],model,meta,audio,\'cpu\',return_char_alignments=True)\n        segments.append(dict(parent_segment_id=s[\'id\'],result=result))\n        got=result[\'word_segments\']\n        expected=s[\'text\'].split()\n        if [w[\'word\'] for w in got]!=expected:\n            failures.append(dict(segment_id=s[\'id\'],reason=\'token_sequence_mismatch\',expected=expected,observed=[w[\'word\'] for w in got]))\n            got=[dict(word=w) for w in expected]\n        for j,w0 in enumerate(got):\n            w=dict(w0);w.update(occurrence_id=f"VD-RK-nY66k_0090_a{i:03d}_{j:03d}",parent_segment_id=s[\'id\'],word_index=j)\n            w[\'unsupported_letters\']=[c for c in w[\'word\'] if unicodedata.category(c)[0] in \'LN\' and c.lower() not in meta[\'dictionary\']]\n            w[\'alignment_reasons\']=alignment_reasons(w)\n            w[\'lookup\']=lookup(w[\'word\']);words.append(w)\n        print(f\'ALIGN {i+1}/{len(raw["segments"])} words={len(got)}\',flush=True)\n    # Conflicting alignments at adjacent segment margins cannot both be exported.\n    for before,after in zip(words,words[1:]):\n        if before.get(\'end\') is not None and after.get(\'start\') is not None and before[\'end\']>after[\'start\']+.02:\n            before[\'alignment_reasons\'].append(\'overlapping_neighbor\');after[\'alignment_reasons\'].append(\'overlapping_neighbor\')\n    save(out/\'alignment.json\',dict(model_id=model_id,revision=Path(model_path).name,method=\'whisperx_word_ctc\',words=words,segments=segments,failures=failures))\n    print(\'ALIGN COMPLETE\',len(words),sum(not w[\'alignment_reasons\'] for w in words),flush=True)\n\ndef box(points,w,h,pad=.15):\n    import numpy as np\n    p=np.array(points);lo=p.min(axis=0);hi=p.max(axis=0);delta=(hi-lo)*pad\n    return [max(0,int((lo[0]-delta[0])*w)),max(0,int((lo[1]-delta[1])*h)),min(w,int(math.ceil((hi[0]+delta[0])*w))),min(h,int(math.ceil((hi[1]+delta[1])*h)))]\ndef iou(a,b):\n    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))\n    return inter/max(1,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)\n\ndef visual_stage(root,out):\n    import cv2,numpy as np,mediapipe as mp\n    from mediapipe.tasks import python\n    from mediapipe.tasks.python import vision\n    assets=out/\'assets\';assets.mkdir(exist_ok=True)\n    task=download(\'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task\',assets/\'face_landmarker.task\')\n    detector=vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(base_options=python.BaseOptions(model_asset_path=str(task)),running_mode=vision.RunningMode.IMAGE,num_faces=1,output_facial_transformation_matrixes=True))\n    yunet_path=download(\'https://media.githubusercontent.com/media/opencv/opencv_zoo/f12e12798e8314f7c074a6656816c048dcc95b7a/models/face_detection_yunet/face_detection_yunet_2023mar.onnx\',assets/\'face_detection_yunet_2023mar.onnx\')\n    cv2.setNumThreads(2)\n    yunet=cv2.FaceDetectorYN.create(str(yunet_path),\'\',(960,540),CONFIG[\'face_detection_score\'],.3,5000)\n    cap=cv2.VideoCapture(str(root/\'raw/VD-RK-nY66k.mp4\'))\n    assert abs(cap.get(cv2.CAP_PROP_FPS)-30)<.001\n    cap.set(cv2.CAP_PROP_POS_FRAMES,2700)\n    cuts=[round((x-90)*30) for x in read(root/\'pilot_0090_0390/scene_candidates.json\')[\'source_seconds\']]\n    frames=[]; track=-1;prev=None;shot=0\n    lips=[61,146,91,181,84,17,314,405,321,375,291,409,270,269,267,0,37,39,40,185,13,14]\n    for f in range(9000):\n        ok,image=cap.read()\n        if not ok: raise RuntimeError(f\'Video ended at frame {f}\')\n        if f in cuts: shot+=1;prev=None\n        h,w=image.shape[:2]\n        small=cv2.resize(image,(960,540))\n        _,detections=yunet.detect(small)\n        count=0 if detections is None else len(detections)\n        rec=dict(frame=f,source_frame=f+2700,time=f/30,shot=shot,face_count=count,good=False)\n        res=None\n        if count==1:\n            fx,fy,fw,fh=detections[0,:4]*2\n            cx=fx+fw/2;cy=fy+fh/2;side=max(fw,fh)*2\n            ox=max(0,int(cx-side/2));oy=max(0,int(cy-side/2));ex=min(w,int(cx+side/2));ey=min(h,int(cy+side/2))\n            roi=image[oy:ey,ox:ex]\n            rec.update(detector_score=float(detections[0,14]),landmark_input_xyxy=[ox,oy,ex,ey])\n            res=detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=cv2.cvtColor(roi,cv2.COLOR_BGR2RGB)))\n        if res is not None and len(res.face_landmarks)==1:\n            lm=res.face_landmarks[0]; pts=[((p.x*(ex-ox)+ox)/w,(p.y*(ey-oy)+oy)/h) for p in lm]\n            face=box(pts,w,h,0);mouth=box([pts[k] for k in lips],w,h,.35)\n            if prev is None or iou(prev,face)<.3:track+=1\n            R=np.array(res.facial_transformation_matrixes[0])[:3,:3]\n            yaw=math.degrees(math.atan2(-R[2,0],math.hypot(R[0,0],R[1,0])))\n            width=float(np.linalg.norm((np.array(pts[61])-np.array(pts[291]))*[w,h]))\n            x1,y1,x2,y2=mouth;roi=image[y1:y2,x1:x2]\n            blur=float(cv2.Laplacian(cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var()) if roi.size else 0\n            jump=0 if prev is None else 1-iou(prev,face)\n            reasons=[]\n            if width<CONFIG[\'min_mouth_pixels\'] or face[3]-face[1]<CONFIG[\'min_face_height\']:reasons.append(\'small_face_or_mouth\')\n            if abs(yaw)>CONFIG[\'max_yaw_degrees\']:reasons.append(\'head_pose\')\n            if blur<CONFIG[\'min_mouth_laplacian\']:reasons.append(\'blur\')\n            if jump>CONFIG[\'max_landmark_jump\']:reasons.append(\'landmark_jump\')\n            if any(not 0<=pts[k][0]<=1 or not 0<=pts[k][1]<=1 for k in lips):reasons.append(\'mouth_outside_frame\')\n            rec.update(face_box=face,mouth_box=mouth,mouth_width=width,yaw=yaw,blur=blur,track_id=track,reasons=reasons,good=not reasons)\n            prev=face\n        else:rec[\'reasons\']=[\'face_count\' if count!=1 else \'landmarks_missing\'];prev=None\n        frames.append(rec)\n        if f%900==0:print(\'VISUAL\',f,\'/9000\',flush=True)\n    cap.release();detector.close()\n    save(out/\'visual.json\',dict(frame_rate=30,model_sha256=sha(task),detector_sha256=sha(yunet_path),detector=\'YuNet 2023mar at OpenCV Zoo f12e12798e8314f7c074a6656816c048dcc95b7a; score >= 0.9; MediaPipe on 2x face crop\',frames=frames,limitations=\'landmark, pose and blur proxies; not a dedicated mouth-occlusion classifier; track IDs are local continuity IDs\'))\n    print(\'VISUAL COMPLETE\',sum(f[\'good\'] for f in frames),\'/9000\',flush=True)\n\ndef runs(frames):\n    result=[];start=None;key=None\n    for i,f in enumerate(frames+[{\'good\':False}]):\n        newkey=(f.get(\'shot\'),f.get(\'track_id\')) if f[\'good\'] else None\n        if start is not None and newkey!=key:\n            result.append((start,i,key));start=None\n        if f[\'good\'] and start is None:start=i;key=newkey\n    return result\n\ndef sync_stage(root,out):\n    import cv2,numpy as np,torch,python_speech_features\n    from scipy.io import wavfile\n    from scipy.signal import medfilt\n    torch.set_num_threads(2)\n    assets=out/\'assets\';assets.mkdir(exist_ok=True)\n    # Official implementation, recorded by content hash alongside the checkpoint.\n    modelpy=download(\'https://raw.githubusercontent.com/joonson/syncnet_python/1fbfc081bda41f27aab51c0fbaa0e5e337126cf3/SyncNetModel.py\',assets/\'SyncNetModel.py\')\n    weights=download(\'https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/syncnet_v2.model\',assets/\'syncnet_v2.model\')\n    sys.path.insert(0,str(assets));from SyncNetModel import S\n    model=S();model.load_state_dict(torch.load(weights,map_location=\'cpu\',weights_only=True));model.eval()\n    frames=read(out/\'visual.json\')[\'frames\']; _,wav=wavfile.read(root/\'pilot_0090_0390/audio_16k.wav\')\n    cap=cv2.VideoCapture(str(root/\'raw/VD-RK-nY66k.mp4\'));results=[]\n    eligible=[r for r in runs(frames) if r[1]-r[0]>=90]\n    for n,(a,b,key) in enumerate(eligible):\n        # Divide continuous suitable views into >=3-second windows, never across a cut.\n        edges=list(range(a,b,180))\n        if len(edges)>1 and b-edges[-1]<90:edges.pop()\n        edges.append(b)\n        for begin,end in zip(edges,edges[1:]):\n            start=begin/30;duration=(end-begin)/30\n            nf=int(math.floor(duration*25));indices=begin+np.floor(np.arange(nf)*30/25+.5).astype(int)\n            boxes=np.array([frames[int(i)][\'face_box\'] for i in indices],dtype=float)\n            smooth=np.stack([medfilt(boxes[:,j],13) for j in range(4)],axis=1)\n            crops=[]\n            cap.set(cv2.CAP_PROP_POS_FRAMES,2700+begin);cursor=begin-1;im=None\n            for idx,bb in zip(indices,smooth):\n                while cursor<idx:\n                    ok,im=cap.read();cursor+=1\n                    if not ok:raise RuntimeError(\'Missing sync frame\')\n                x=(bb[0]+bb[2])/2;y=(bb[1]+bb[3])/2;s=max(bb[2]-bb[0],bb[3]-bb[1])/2\n                pad=int(s*1.8)+2;im2=np.pad(im,((pad,pad),(pad,pad),(0,0)),constant_values=110)\n                crop=im2[int(y+pad-s):int(y+pad+s*1.8),int(x+pad-s*1.4):int(x+pad+s*1.4)]\n                crops.append(cv2.resize(crop,(224,224)))\n            video=torch.from_numpy(np.array(crops).transpose(3,0,1,2)[None].astype(np.float32))\n            audio=wav[round(start*16000):round((start+nf/25)*16000)]\n            mfcc=python_speech_features.mfcc(audio,16000).T\n            sound=torch.from_numpy(mfcc[None,None].astype(np.float32));vf=[];af=[]\n            last=min(nf-5,(mfcc.shape[1]-20)//4+1)\n            with torch.inference_mode():\n                for j in range(0,last,16):\n                    vv=torch.cat([video[:,:,k:k+5] for k in range(j,min(last,j+16))])\n                    aa=torch.cat([sound[:,:,:,k*4:k*4+20] for k in range(j,min(last,j+16))])\n                    vf.append(model.forward_lip(vv));af.append(model.forward_aud(aa))\n            vf=torch.cat(vf);af=torch.cat(af);shift=10\n            # Exclude edge padding from the offset comparison.\n            center=vf[shift:-shift]\n            curve=torch.stack([torch.nn.functional.pairwise_distance(center,af[shift+d:len(af)-shift+d]).mean() for d in range(-shift,shift+1)])\n            best=int(curve.argmin());offset=shift-best;confidence=float(curve.median()-curve[best])\n            passed=confidence>=CONFIG[\'sync_confidence\'] and abs(offset)<=CONFIG[\'max_sync_offset_frames\']\n            rec=dict(start=start,end=end/30,shot=key[0],track_id=key[1],offset_frames_25fps=offset,confidence=confidence,distance_curve=curve.tolist(),pass_gate=passed,method=\'SyncNet face/audio correspondence proxy\')\n            results.append(rec);save(out/\'sync.partial.json\',results)\n            print(f\'SYNC track {n+1}/{len(eligible)} {start:.2f}-{end/30:.2f} offset={offset} confidence={confidence:.2f} pass={passed}\',flush=True)\n    cap.release()\n    save(out/\'sync.json\',dict(windows=results,model_sha256=sha(weights),implementation_sha256=sha(modelpy),method_notes=\'Official SyncNet network; MediaPipe crops with official crop geometry; nearest source frames sampled at 25 fps, no time warp; mean distance excludes shift-padding edges. Provisional thresholds; no independent identity verification.\'))\n\ndef union_box(boxes,pad,w=1920,h=1080):\n    x1=min(b[0] for b in boxes);y1=min(b[1] for b in boxes);x2=max(b[2] for b in boxes);y2=max(b[3] for b in boxes)\n    dx=(x2-x1)*pad;dy=(y2-y1)*pad\n    x=max(0,int(x1-dx)//2*2);y=max(0,int(y1-dy)//2*2)\n    right=min(w,math.ceil((x2+dx)/2)*2);bottom=min(h,math.ceil((y2+dy)/2)*2)\n    return [x,y,right-x,bottom-y]\n\ndef export_stage(root,out):\n    import imageio_ffmpeg,sqlite3,zipfile\n    ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()\n    alignment=read(out/\'alignment.json\');frames=read(out/\'visual.json\')[\'frames\'];windows=read(out/\'sync.json\')[\'windows\']\n    clips=out/\'clips\';clips.mkdir(exist_ok=True);entries=[]\n    for k,w in enumerate(alignment[\'words\']):\n        reasons=list(w[\'alignment_reasons\']);r=dict(w);r[\'previous_word\']=alignment[\'words\'][k-1][\'word\'] if k else None;r[\'next_word\']=alignment[\'words\'][k+1][\'word\'] if k+1<len(alignment[\'words\']) else None\n        if not reasons:\n            a,b=cut_window(w[\'start\'],w[\'end\']);first=round(a*30);last=round(b*30);fs=frames[first:last]\n            if not fs or not all(f[\'good\'] for f in fs):reasons.append(\'visual_quality\')\n            if len({(f[\'shot\'],f.get(\'track_id\')) for f in fs})!=1:reasons.append(\'shot_or_track_change\')\n            matching=[i for i,s in enumerate(windows) if s[\'pass_gate\'] and s[\'start\']<=a+1e-6 and s[\'end\']>=b-1e-6]\n            if not matching:reasons.append(\'sync_unconfirmed_or_failed\')\n            r.update(cut_start=a,cut_end=b,source_start=90+a,source_end=90+b,lexical_offset_start=w[\'start\']-a,lexical_offset_end=w[\'end\']-a)\n            if not reasons:\n                face=union_box([f[\'face_box\'] for f in fs],.2);mouth=union_box([f[\'mouth_box\'] for f in fs],.05)\n                stem=clips/w[\'occurrence_id\'];outputs={x:str(stem.name)+\'_\'+x+\'.mp4\' for x in [\'av\',\'face\',\'mouth\']};outputs[\'audio\']=stem.name+\'_audio.wav\'\n                fc=f\'[0:v]split=3[v0][v1][v2];[v1]crop={face[2]}:{face[3]}:{face[0]}:{face[1]},scale=256:256:force_original_aspect_ratio=decrease:force_divisible_by=2,pad=256:256:(ow-iw)/2:(oh-ih)/2[vf];[v2]crop={mouth[2]}:{mouth[3]}:{mouth[0]}:{mouth[1]},scale=160:96:force_original_aspect_ratio=decrease:force_divisible_by=2,pad=160:96:(ow-iw)/2:(oh-ih)/2[vm]\'\n                cmd=[ffmpeg,\'-y\',\'-v\',\'error\',\'-ss\',str(90+a),\'-t\',str(b-a),\'-i\',str(root/\'raw/VD-RK-nY66k.mp4\'),\'-filter_complex\',fc]\n                for label,tag in [(\'v0\',\'av\'),(\'vf\',\'face\'),(\'vm\',\'mouth\')]:\n                    cmd+=[\'-map\',f\'[{label}]\',\'-map\',\'0:a:0\',\'-c:v\',\'libx264\',\'-preset\',\'veryfast\',\'-crf\',\'20\',\'-threads\',\'2\',\'-c:a\',\'aac\',\'-b:a\',\'128k\',\'-movflags\',\'+faststart\',str(clips/outputs[tag])]\n                cmd+=[\'-map\',\'0:a:0\',\'-ac\',\'1\',\'-ar\',\'16000\',\'-c:a\',\'pcm_s16le\',str(clips/outputs[\'audio\'])]\n                subprocess.run(cmd,check=True)\n                r.update(files={t:\'clips/\'+name for t,name in outputs.items()},face_crop_xywh=face,mouth_crop_xywh=mouth,crop_resize=\'aspect-preserving scale and centered black padding\',face_output_size=[256,256],mouth_output_size=[160,96],track_id=fs[0][\'track_id\'],shot=fs[0][\'shot\'],sync_window=matching[0],status=\'automatic_candidate_pass\',phonemes=None)\n        r[\'rejection_reasons\']=reasons\n        if reasons:r[\'status\']=\'excluded\'\n        entries.append(r)\n        if k%50==0:print(\'EXPORT\',k,\'/\',len(alignment[\'words\']),\'passing\',sum(x[\'status\']==\'automatic_candidate_pass\' for x in entries),flush=True)\n    save(out/\'occurrences.json\',entries)\n    dictionary={}\n    for r in entries:\n        if r[\'status\']==\'automatic_candidate_pass\':dictionary.setdefault(r[\'lookup\'],[]).append(r[\'occurrence_id\'])\n    save(out/\'dictionary.json\',dictionary)\n    db=sqlite3.connect(out/\'dictionary.sqlite\');db.execute(\'CREATE TABLE IF NOT EXISTS occurrences (id TEXT PRIMARY KEY, word TEXT, lookup TEXT, status TEXT, metadata TEXT)\');db.execute(\'DELETE FROM occurrences\')\n    db.executemany(\'INSERT INTO occurrences VALUES (?,?,?,?,?)\',[(r[\'occurrence_id\'],r[\'word\'],r[\'lookup\'],r[\'status\'],json.dumps(r,ensure_ascii=False)) for r in entries]);db.execute(\'CREATE INDEX IF NOT EXISTS word_lookup ON occurrences(lookup,status)\');db.commit();db.close()\n    kept=[r for r in entries if r[\'status\']==\'automatic_candidate_pass\'];intervals=sorted((r[\'cut_start\'],r[\'cut_end\']) for r in kept);merged=[]\n    for a,b in intervals:\n        if merged and a<=merged[-1][1]:merged[-1][1]=max(b,merged[-1][1])\n        else:merged.append([a,b])\n    report=dict(input_seconds=300,aligned_entries=len(entries),alignment_pass=sum(not r[\'alignment_reasons\'] for r in entries),visual_good_frames=sum(f[\'good\'] for f in frames),total_frames=len(frames),sync_windows=len(windows),sync_windows_pass=sum(w[\'pass_gate\'] for w in windows),exported_occurrences=len(kept),vocabulary=len(dictionary),retained_unique_seconds=sum(b-a for a,b in merged),exported_seconds_with_repetition=sum(r[\'cut_end\']-r[\'cut_start\'] for r in kept),exclusion_reasons=dict(collections.Counter(x for r in entries for x in r[\'rejection_reasons\'])),thresholds=CONFIG,status=\'automatically filtered pilot candidates; no measured human-reference accuracy\',scope=\'Word corpus only. Phonemes, canonical generation, composition and dubbing not run.\')\n    save(out/\'report.json\',report);print(json.dumps(report,indent=2),flush=True)\n    with zipfile.ZipFile(root/\'urdu_pilot_processed.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in out.rglob(\'*\'):\n            if p.is_file() and \'assets\' not in p.relative_to(out).parts and \'.partial.\' not in p.name:z.write(p,p.relative_to(out))\n    print(\'BUNDLE\',root/\'urdu_pilot_processed.zip\',flush=True)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'align\',\'visual\',\'sync\',\'export\']);p.add_argument(\'--root\',type=Path,default=Path(\'/content/urdu-av-pilot\'));args=p.parse_args()\n    out=args.root/\'processed_0090_0390\';out.mkdir(exist_ok=True)\n    save(out/\'config.json\',CONFIG)\n    globals()[args.stage+\'_stage\'](args.root,out)\nif __name__==\'__main__\': main()\n')


In [ ]:
Path('/content/finalize_pilot.py').write_text('"""Verify exported files, record provenance, and make an offline word browser."""\nfrom pathlib import Path\nimport argparse, hashlib, html, json, subprocess, sys, wave, zipfile\nfrom datetime import datetime, timezone\nfrom importlib.metadata import version, PackageNotFoundError\n\n\ndef read(path): return json.loads(path.read_text())\ndef save(path, value): path.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\ndef sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef finalize(root, pipeline):\n    import imageio_ffmpeg\n    import numpy as np\n    from scipy.io import wavfile\n    from scipy.signal import correlate, resample_poly\n    out=root/\'processed_0090_0390\'\n    entries=read(out/\'occurrences.json\')\n    kept=[r for r in entries if r[\'status\']==\'automatic_candidate_pass\']\n    sr,reference=wavfile.read(root/\'pilot_0090_0390/audio_16k.wav\')\n    assert sr==16000\n    checks=[]\n    ffmpeg=imageio_ffmpeg.get_ffmpeg_exe()\n    for i,r in enumerate(kept):\n        expected=r[\'cut_end\']-r[\'cut_start\'];expected_frames=round(expected*30)\n        streams=[]\n        for kind,relative in r[\'files\'].items():\n            path=out/relative\n            assert path.is_file() and path.stat().st_size>0\n            if kind==\'audio\':\n                with wave.open(str(path)) as w:\n                    assert w.getframerate()==16000 and w.getnchannels()==1\n                    duration=w.getnframes()/16000\n                assert abs(duration-expected)<=1/16000+1e-6, (path,duration,expected)\n                _,sample=wavfile.read(path)\n                # Independent timing check against the authoritative processing WAV.\n                center=round(r[\'cut_start\']*16000);lo=max(0,center-1600)\n                region=reference[lo:min(len(reference),center+len(sample)+1600)].astype(float)\n                # Cuts on the 30 fps grid can fall between 16 kHz samples.\n                # Compare on a 48 kHz analysis grid without modifying saved media.\n                region=resample_poly(region,3,1);y=resample_poly(sample.astype(float),3,1)\n                y-=y.mean();region-=region.mean()\n                cross=correlate(region,y,mode=\'valid\',method=\'fft\')\n                at=int(np.argmax(cross));match=region[at:at+len(y)];match=match-match.mean()\n                similarity=float(np.dot(match,y)/(np.linalg.norm(match)*np.linalg.norm(y)+1e-12))\n                lag=(lo+at/3-center)/16000\n                assert abs(lag)<=.002 and similarity>.98, (r[\'occurrence_id\'],lag,similarity)\n                streams.append(dict(kind=kind,duration=duration,reference_lag_seconds=lag,reference_correlation=similarity,comparison_sample_rate=48000,sha256=sha(path)))\n            else:\n                probe=json.loads(subprocess.check_output([\'ffprobe\',\'-v\',\'error\',\'-count_frames\',\'-show_streams\',\'-of\',\'json\',str(path)],text=True))\n                video=[s for s in probe[\'streams\'] if s[\'codec_type\']==\'video\'][0]\n                audio=[s for s in probe[\'streams\'] if s[\'codec_type\']==\'audio\'][0]\n                assert int(video[\'nb_read_frames\'])==expected_frames,(path,video[\'nb_read_frames\'],expected_frames)\n                assert abs(float(video[\'duration\'])-expected)<1e-5\n                assert abs(float(audio[\'duration\'])-expected)<.04\n                assert abs(float(video.get(\'start_time\',0))-float(audio.get(\'start_time\',0)))<.04\n                # Decode both streams completely, not just container headers.\n                subprocess.run([ffmpeg,\'-v\',\'error\',\'-i\',str(path),\'-map\',\'0:v:0\',\'-map\',\'0:a:0\',\'-f\',\'null\',\'-\'],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)\n                streams.append(dict(kind=kind,video_frames=int(video[\'nb_read_frames\']),video_duration=float(video[\'duration\']),audio_duration=float(audio[\'duration\']),width=video[\'width\'],height=video[\'height\'],sha256=sha(path)))\n        checks.append(dict(occurrence_id=r[\'occurrence_id\'],files=streams,status=\'passed\'))\n        if i%25==0:print(\'VERIFY\',i,\'/\',len(kept),flush=True)\n    save(out/\'export_verification.json\',dict(verified_occurrences=len(checks),file_count=sum(len(c[\'files\']) for c in checks),checks=checks,scope=\'File decodability, frame counts, duration, stream starts and WAV source-time correlation on a 48 kHz comparison grid (saved WAVs remain 16 kHz). Not a perceptual lip-sync or word-boundary accuracy measurement.\'))\n    versions={}\n    for name in [\'torch\',\'torchaudio\',\'transformers\',\'whisperx\',\'numpy\',\'scipy\',\'mediapipe\',\'opencv-python\',\'opencv-contrib-python\',\'python_speech_features\',\'imageio-ffmpeg\',\'huggingface_hub\']:\n        try:versions[name]=version(name)\n        except PackageNotFoundError:pass\n    manifest=dict(created_at_utc=datetime.now(timezone.utc).isoformat(),source_id=\'VD-RK-nY66k\',source_url=\'https://www.youtube.com/watch?v=VD-RK-nY66k\',source_interval=[90,390],language=\'ur\',transcript_selection=\'Unchanged ASR text accepted by user on 2026-09-27; acceptance does not validate timing\',source_sha256=sha(root/\'raw/VD-RK-nY66k.mp4\'),transcript_sha256=sha(root/\'pilot_0090_0390/transcript_raw.json\'),pipeline_sha256=sha(pipeline),python=sys.version,packages=versions,speaker_label=\'Presenter attributed in source title to Dr. Arfa Zehra; no biometric identity verification\',time_mapping=\'Source time = pilot time + 90 seconds; no temporal warp\',rights=\'Dataset redistribution permission not established\',threshold_status=\'Provisional and not calibrated against an independent labeled set\')\n    save(out/\'run_manifest.json\',manifest)\n    (out/\'pilot_pipeline.py\').write_bytes(pipeline.read_bytes())\n    (out/\'finalize_pilot.py\').write_bytes(Path(__file__).read_bytes())\n    # Embed metadata so the browser works from disk without fetch/CORS requirements.\n    payload=json.dumps(kept,ensure_ascii=False).replace(\'<\',\'\\\\u003c\')\n    page=\'\'\'<!doctype html><meta charset="utf-8"><title>Urdu pilot word dictionary</title>\n<style>body{font:17px system-ui;max-width:1100px;margin:30px auto;padding:0 20px;background:#f7f8fc;color:#20283a}input{font:inherit;padding:12px;width:85%}article{background:white;padding:18px;margin:14px 0;border:1px solid #dde2ec;border-radius:9px}video{width:32%;max-height:240px;background:#111}small{display:block}a{margin-right:15px}h2{font-size:26px}</style>\n<h1>Urdu pilot word dictionary</h1><p>Automatically filtered candidates from source 01:30–06:30. Search Urdu text or an occurrence ID. Repeated words remain separate. Clips include 120 ms of context, rounded to source video frames.</p>\n<p>Scores and boundaries are model estimates. No phoneme labels, measured accuracy, or identity transfer. Frame checks and SyncNet use provisional thresholds.</p>\n<input id="q" placeholder="Search, e.g. اردو" aria-label="Search Urdu word"><p id="count"></p><main id="results"></main>\n<script>const rows=PAYLOAD;const q=document.getElementById(\'q\');const area=document.getElementById(\'results\');\nfunction show(){const term=q.value.trim();const matches=rows.filter(r=>!term||r.word.includes(term)||r.occurrence_id.includes(term));document.getElementById(\'count\').textContent=matches.length+\' matches; showing first 30\';area.replaceChildren();for(const r of matches.slice(0,30)){const a=document.createElement(\'article\');const h=document.createElement(\'h2\');h.textContent=r.word;h.dir=\'rtl\';a.append(h);const p=document.createElement(\'p\');p.textContent=r.occurrence_id+\' | source \'+r.source_start.toFixed(3)+\'–\'+r.source_end.toFixed(3)+\' s | word inside clip \'+r.lexical_offset_start.toFixed(3)+\'–\'+r.lexical_offset_end.toFixed(3)+\' s\';a.append(p);for(const kind of [\'av\',\'face\',\'mouth\']){const v=document.createElement(\'video\');v.controls=true;v.preload=\'none\';v.src=r.files[kind];v.title=kind;a.append(v);}const links=document.createElement(\'p\');for(const [kind,path]of Object.entries(r.files)){const l=document.createElement(\'a\');l.href=path;l.textContent=kind;links.append(l);}a.append(links);area.append(a);}}\nq.addEventListener(\'input\',show);show();</script>\'\'\'.replace(\'PAYLOAD\',payload)\n    (out/\'dictionary.html\').write_text(page)\n    (out/\'README.txt\').write_text(\'Open dictionary.html next to clips/ in a regular browser. dictionary.json maps written forms to all passing occurrences; occurrences.json and dictionary.sqlite retain excluded entries and reasons. run_manifest.json contains source and model-run provenance. report.json contains automatic counts. export_verification.json contains file integrity/timing checks. Model estimates are not independent annotation ground truth. No media time stretching, waveform averaging or pixel averaging was applied.\\n\')\n    bundle=root/\'urdu_pilot_processed.zip\'\n    with zipfile.ZipFile(bundle,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in out.rglob(\'*\'):\n            if p.is_file() and \'assets\' not in p.relative_to(out).parts and \'.partial.\' not in p.name:z.write(p,p.relative_to(out))\n    print(\'VERIFIED\',len(checks),\'occurrences;\',bundle,\'bytes\',bundle.stat().st_size,flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',type=Path,default=Path(\'/content/urdu-av-pilot\'));p.add_argument(\'--pipeline\',type=Path,default=Path(\'/content/pilot_pipeline.py\'));a=p.parse_args();finalize(a.root,a.pipeline)\n')


In [ ]:
def run_script(script, *arguments):
    with subprocess.Popen([sys.executable,'-u','/content/'+script,*arguments],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
        for line in process.stdout: print(line,end='',flush=True)
        result=process.wait()
    if result: raise RuntimeError(f'{script} {arguments} failed with exit {result}')
for stage in ['align','visual','sync','export']:
    print('START',stage,flush=True)
    run_script('pilot_pipeline.py',stage)
run_script('finalize_pilot.py')


In [ ]:
import json
print(json.dumps(json.loads((ROOT/'processed_0090_0390/report.json').read_text()),ensure_ascii=False,indent=2))
from google.colab import files
files.download(str(ROOT/'urdu_pilot_processed.zip'))


Open `dictionary.html` beside `clips/` in a regular browser. `dictionary.sqlite` and `occurrences.json` include exclusion reasons. Keep the complete ZIP before the temporary runtime expires. Original recordings and predicted annotations remain separate. Redistribution permission is not established for this TEDx source.